In [ ]:
Ниже LightGBM и графики с результатом 9,11

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# КОНФИГ
# =============================================================================
TRAIN_PATH = 'train_dataset.csv'
VALID_PATH = 'valid_features.csv'
OUTPUT_VALID = 'solverdata.csv'

TARGET = 'Выработка. Результирующий расчет'
DT_COL = 'METEOFORECASTHOUR_OPENM_Datetime'

CAPACITY    = 90.09
N_TURBINES  = 26

# Бленд сидов (как в твоём коде)
SEEDS = [42, 123, 777, 2024, 9001]

# =============================================================================
# 1. ЗАГРУЗКА ДАННЫХ
# =============================================================================
print("=" * 60)
print("Загрузка данных...")
train = pd.read_csv(TRAIN_PATH)
valid = pd.read_csv(VALID_PATH)

train[DT_COL] = pd.to_datetime(train[DT_COL])
valid[DT_COL] = pd.to_datetime(valid[DT_COL])

valid_order = valid[DT_COL].copy()
train = train.sort_values(DT_COL).reset_index(drop=True)

print(f"Train: {train.shape}, период {train[DT_COL].min()} ... {train[DT_COL].max()}")
print(f"Valid: {valid.shape}, период {valid[DT_COL].min()} ... {valid[DT_COL].max()}")

# =============================================================================
# 2. ИНТЕРПОЛЯЦИЯ НА РЕАЛЬНУЮ ВЫСОТУ РОТОРА 84 м
# =============================================================================
HUB_H, LOW_H, HIGH_H = 84.0, 80.0, 120.0
for df in (train, valid):
    # Скорость (степенной закон)
    ratio = df['wind_speed_120m'] / df['wind_speed_80m'].clip(lower=0.1)
    alpha = np.log(ratio.clip(lower=0.1)) / np.log(HIGH_H / LOW_H)
    df['wind_speed_84m'] = df['wind_speed_80m'] * (HUB_H / LOW_H) ** alpha

    # Направление (линейная интерполяция sin/cos, результат нормируем [0,1])
    sin_low, cos_low = np.sin(2 * np.pi * df['wind_direction_80m']), np.cos(2 * np.pi * df['wind_direction_80m'])
    sin_high, cos_high = np.sin(2 * np.pi * df['wind_direction_120m']), np.cos(2 * np.pi * df['wind_direction_120m'])
    w_low = (HIGH_H - HUB_H) / (HIGH_H - LOW_H)      # 0.9
    sin_84 = sin_low * w_low + sin_high * (1 - w_low)
    cos_84 = cos_low * w_low + cos_high * (1 - w_low)
    df['wind_direction_84m'] = np.arctan2(sin_84, cos_84) / (2 * np.pi) % 1.0   # [0,1]

    # Температура (линейная)
    if 'temperature_120m' in df.columns:
        df['temperature_84m'] = df['temperature_80m'] * w_low + df['temperature_120m'] * (1 - w_low)
    else:
        df['temperature_84m'] = df['temperature_80m']

# =============================================================================
# 3. ФИЧЕ-ИНЖИНИРИНГ (объединяет лучшие признаки из обоих подходов)
# =============================================================================
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dt = df[DT_COL]

    # ---- Временные (расширенный набор из твоего кода) ----
    df['dayofyear'] = dt.dt.dayofyear
    df['dayofweek'] = dt.dt.dayofweek
    df['week']      = dt.dt.isocalendar().week.astype(int)
    df['hour_sin']  = np.sin(2 * np.pi * df['hour_of_day'] / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour_of_day'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['doy_sin']   = np.sin(2 * np.pi * df['dayofyear'] / 365)
    df['doy_cos']   = np.cos(2 * np.pi * df['dayofyear'] / 365)

    # ---- Направления [0,1] в синус/косинус (твоя сильная сторона) ----
    for col in ['wind_direction_10m', 'wind_direction_80m',
                'wind_direction_120m', 'wind_direction_180m']:
        df[col + '_sin'] = np.sin(2 * np.pi * df[col])
        df[col + '_cos'] = np.cos(2 * np.pi * df[col])

    # ---- Статические признаки (оставляем всё, что было у тебя) ----
    df['wind_speed_80m_sq']    = df['wind_speed_80m'] ** 2
    df['wind_speed_80m_cube']  = df['wind_speed_80m'] ** 3
    df['wind_speed_120m_cube'] = df['wind_speed_120m'] ** 3
    df['wind_shear']           = df['wind_speed_120m'] - df['wind_speed_10m']
    df['gust_ratio']           = df['wind_gusts_10m'] / (df['wind_speed_10m'] + 0.1)
    df['wind_avg']             = df[['wind_speed_80m', 'wind_speed_120m']].mean(axis=1)

    # ---- НОВЫЕ ПРИЗНАКИ НА ОСНОВЕ 84 м (наши лучшие находки) ----
    df['ws84_cubed'] = df['wind_speed_84m'] ** 3
    wd = 2 * np.pi * df['wind_direction_84m']
    df['u84'] = df['wind_speed_84m'] * np.cos(wd)
    df['v84'] = df['wind_speed_84m'] * np.sin(wd)

    if 'pressure_msl' in df.columns and 'temperature_84m' in df.columns:
        df['air_density'] = df['pressure_msl'] * 100 / (287.05 * (df['temperature_84m'] + 273.15))
        df['wind_power_density'] = 0.5 * df['air_density'] * df['ws84_cubed']

    if 'wind_speed_120m' in df.columns:
        df['shear_120_84'] = df['wind_speed_120m'] - df['wind_speed_84m']
    if 'wind_speed_10m' in df.columns:
        df['shear_84_10'] = df['wind_speed_84m'] - df['wind_speed_10m']

    # Теоретическая мощность станции
    ws = df['wind_speed_84m'].values
    theory = np.zeros_like(ws)
    reg = (ws >= 3.0) & (ws < 10.3)
    theory[reg] = 90.09 * ((ws[reg] - 3.0) / (10.3 - 3.0))**3
    rated = (ws >= 10.3) & (ws <= 25.0)
    theory[rated] = 90.09
    df['theory_power_total'] = theory

    # Лаги скорости (1-3 ч) и направления (1 ч)
    for lag in [1, 2, 3]:
        df[f'ws84_lag{lag}'] = df['wind_speed_84m'].shift(lag)
    wd_lag = df['wind_direction_84m'].shift(1)
    df['wd84_lag1_sin'] = np.sin(2 * np.pi * wd_lag)
    df['wd84_lag1_cos'] = np.cos(2 * np.pi * wd_lag)

    # ---- Обработка ремонтов (умная + твоя защита) ----
    df['repair_ma6']  = df['Кол-во_ВЭУ_в_ремонте'].rolling(6, min_periods=1).mean()
    df['has_repair']  = (df['Кол-во_ВЭУ_в_ремонте'] > 0).astype(int)
    df['turbines_available'] = 26 - df['Кол-во_ВЭУ_в_ремонте']
    df['available_ratio']    = df['turbines_available'] / 26

    return df

def fill_missing_180m(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['wind_speed_180m']         = df['wind_speed_180m'].fillna(df['wind_speed_120m'])
    df['wind_direction_180m']     = df['wind_direction_180m'].fillna(df['wind_direction_120m'])
    df['wind_direction_180m_sin'] = df['wind_direction_180m_sin'].fillna(df['wind_direction_120m_sin'])
    df['wind_direction_180m_cos'] = df['wind_direction_180m_cos'].fillna(df['wind_direction_120m_cos'])
    return df

print("\nГенерация признаков...")
train_f = fill_missing_180m(add_features(train))
valid_f = fill_missing_180m(add_features(valid))

# Удаляем первые строки train из-за лагов, valid заполняем медианой
lag_cols = [c for c in train_f.columns if 'lag' in c]
train_f = train_f.dropna(subset=lag_cols)
valid_f[lag_cols] = valid_f[lag_cols].fillna(train_f[lag_cols].median())

# Исключаем сырой признак ремонта
EXCLUDE_FEATURES = [TARGET, DT_COL, 'Кол-во_ВЭУ_в_ремонте']
FEATURES = [c for c in train_f.columns if c not in EXCLUDE_FEATURES]
print(f"Признаков в модели: {len(FEATURES)}")

X       = train_f[FEATURES].values
y       = train_f[TARGET].values
X_valid = valid_f[FEATURES].values

# =============================================================================
# 4. HOLDOUT-ВАЛИДАЦИЯ (янв-март 2025 — сезонный подход)
# =============================================================================
print("\n" + "=" * 60)
print("Holdout-валидация на Jan-Mar 2025...")
holdout_mask = (train_f[DT_COL] >= '2025-01-01') & (train_f[DT_COL] < '2025-04-01')
X_tr, X_ho = X[~holdout_mask], X[holdout_mask]
y_tr, y_ho = y[~holdout_mask], y[holdout_mask]

holdout_preds, best_iters = [], []
for i, seed in enumerate(SEEDS, 1):
    print(f"  [{i}/{len(SEEDS)}] обучение seed={seed} ...")
    m = lgb.LGBMRegressor(
        n_estimators=3000, learning_rate=0.025, num_leaves=63,
        min_child_samples=20, subsample=0.85, colsample_bytree=0.85,
        reg_alpha=0.1, reg_lambda=0.1,
        random_state=seed, n_jobs=-1, verbose=-1,
    )
    m.fit(X_tr, y_tr, eval_set=[(X_ho, y_ho)],
          callbacks=[lgb.early_stopping(75, verbose=False)])
    holdout_preds.append(np.clip(m.predict(X_ho), 0, CAPACITY))
    best_iters.append(m.best_iteration_ or 1500)

ho_pred = np.mean(holdout_preds, axis=0)
mae   = mean_absolute_error(y_ho, ho_pred)
rmse  = np.sqrt(mean_squared_error(y_ho, ho_pred))
r2    = r2_score(y_ho, ho_pred)
nmae  = 100 * mae / CAPACITY
bias  = np.mean(ho_pred - y_ho)

print(f"\nМетрики holdout:")
print(f"  MAE: {mae:.3f} МВт ({nmae:.2f}%), RMSE: {rmse:.3f} МВт, R²: {r2:.4f}, Bias: {bias:+.3f} МВт")

# График holdout (опционально, оставил для наглядности)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_ho, ho_pred, alpha=0.2, s=8)
axes[0].plot([0, CAPACITY], [0, CAPACITY], 'r--')
axes[0].set_xlabel('Факт, МВт'); axes[0].set_ylabel('Прогноз, МВт')
axes[0].set_title(f'Факт vs Прогноз (R²={r2:.3f}, MAE={mae:.2f} МВт)')
axes[0].grid(alpha=0.3); axes[0].set_aspect('equal')
errors = ho_pred - y_ho
axes[1].hist(errors, bins=60, color='#1f77b4', alpha=0.7, edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Ошибка, МВт'); axes[1].set_ylabel('Частота')
axes[1].set_title(f'Распределение ошибок (σ={errors.std():.2f} МВт)')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('holdout_quality.png', dpi=120)
plt.show()

n_iter = int(np.median(best_iters) * 1.1)

# =============================================================================
# 5. ФИНАЛЬНЫЕ МОДЕЛИ + ПРОГНОЗ НА VALID
# =============================================================================
print("\n" + "=" * 60)
print("Тренировка финальных моделей на всём train...")
final_models, valid_preds = [], []
for i, seed in enumerate(SEEDS, 1):
    print(f"  [{i}/{len(SEEDS)}] финальная модель seed={seed} ...")
    m = lgb.LGBMRegressor(
        n_estimators=n_iter, learning_rate=0.025, num_leaves=63,
        min_child_samples=20, subsample=0.85, colsample_bytree=0.85,
        reg_alpha=0.1, reg_lambda=0.1,
        random_state=seed, n_jobs=-1, verbose=-1,
    )
    m.fit(X, y)
    final_models.append(m)
    valid_preds.append(np.clip(m.predict(X_valid), 0, CAPACITY))

final_pred = np.mean(valid_preds, axis=0)

# Защита по доступной мощности (твоя фишка)
zero_mask = valid_f['turbines_available'].values <= 0
final_pred[zero_mask] = 0.0
final_pred = np.minimum(final_pred, CAPACITY * valid_f['available_ratio'].values + 1e-3)
final_pred = np.clip(final_pred, 0, CAPACITY)

# Сохраняем в порядке исходного valid
result = (pd.DataFrame({DT_COL: valid_f[DT_COL].values, TARGET: final_pred})
            .set_index(DT_COL).loc[valid_order].reset_index())
result[[TARGET]].to_csv(OUTPUT_VALID, index=False)
print(f"\n[OK] Сохранён прогноз для лидерборда: {OUTPUT_VALID} ({len(result)} строк)")

Ниже нейросеть - попробовать (надо настроить GPU)

In [1]:
import pandas as pd
import numpy as np
import requests
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# КОНФИГ
# =============================================================================
TRAIN_PATH = 'train_dataset.csv'
VALID_PATH = 'valid_features.csv'
OUTPUT_VALID = 'solverdata.csv'

TARGET = 'Выработка. Результирующий расчет'
DT_COL = 'METEOFORECASTHOUR_OPENM_Datetime'

CAPACITY    = 90.09
N_TURBINES  = 26
LATITUDE    = 46.8268455973
LONGITUDE   = 38.7179393185

SEEDS = [42, 123, 777, 2024, 9001]
LOOKBACK = 48  # окно истории для нейросети

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

# =============================================================================
# 1. ЗАГРУЗКА ИСХОДНЫХ ДАННЫХ
# =============================================================================
train = pd.read_csv(TRAIN_PATH)
valid = pd.read_csv(VALID_PATH)
train[DT_COL] = pd.to_datetime(train[DT_COL])
valid[DT_COL] = pd.to_datetime(valid[DT_COL])
valid_order = valid[DT_COL].copy()
train = train.sort_values(DT_COL).reset_index(drop=True)

# =============================================================================
# 2. ЗАГРУЗКА ERA5 ДЛЯ TRAIN И VALID
# =============================================================================
def fetch_era5(start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/era5"
    params = {
        "latitude": LATITUDE, "longitude": LONGITUDE,
        "start_date": start_date, "end_date": end_date,
        "hourly": [
            "wind_speed_10m", "wind_speed_100m",
            "wind_direction_10m", "wind_direction_100m",
            "temperature_2m", "pressure_msl"
        ],
        "timezone": "UTC"
    }
    r = requests.get(url, params=params).json()["hourly"]
    df = pd.DataFrame({
        DT_COL: pd.to_datetime(r["time"]),
        "ws10_era5": r["wind_speed_10m"],
        "ws100_era5": r["wind_speed_100m"],
        "wd10_era5": r["wind_direction_10m"],
        "wd100_era5": r["wind_direction_100m"],
        "temp2m_era5": r["temperature_2m"],
        "pressure_era5": r["pressure_msl"],
    })
    return df

print("Загрузка ERA5 для train...")
train_era5 = fetch_era5(train[DT_COL].min().strftime('%Y-%m-%d'),
                        train[DT_COL].max().strftime('%Y-%m-%d'))
print("Загрузка ERA5 для valid...")
valid_era5 = fetch_era5(valid[DT_COL].min().strftime('%Y-%m-%d'),
                        valid[DT_COL].max().strftime('%Y-%m-%d'))

# =============================================================================
# 3. ПРОЦЕССИНГ ERA5 (как в 8.016%)
# =============================================================================
def process_era5(df_era5):
    HUB_H, LOW_H, HIGH_H = 84.0, 10.0, 100.0
    ratio = df_era5['ws100_era5'] / df_era5['ws10_era5'].clip(lower=0.1)
    alpha = np.log(ratio.clip(lower=0.1)) / np.log(HIGH_H / LOW_H)
    df_era5['wind_speed_84m_era5'] = df_era5['ws10_era5'] * (HUB_H / LOW_H) ** alpha

    sin10, cos10 = np.sin(2*np.pi*df_era5['wd10_era5']), np.cos(2*np.pi*df_era5['wd10_era5'])
    sin100, cos100 = np.sin(2*np.pi*df_era5['wd100_era5']), np.cos(2*np.pi*df_era5['wd100_era5'])
    w_low = (HIGH_H - HUB_H) / (HIGH_H - LOW_H)
    sin84 = sin10 * w_low + sin100 * (1 - w_low)
    cos84 = cos10 * w_low + cos100 * (1 - w_low)
    df_era5['wind_direction_84m_era5'] = np.arctan2(sin84, cos84) / (2*np.pi) % 1.0

    df_era5['temperature_84m_era5'] = df_era5['temp2m_era5'] - 0.65 * (84 - 2) / 100
    df_era5['pressure_msl_era5'] = df_era5['pressure_era5']

    df_era5['ws84_cubed_era5'] = df_era5['wind_speed_84m_era5'] ** 3
    wd = 2 * np.pi * df_era5['wind_direction_84m_era5']
    df_era5['u84_era5'] = df_era5['wind_speed_84m_era5'] * np.cos(wd)
    df_era5['v84_era5'] = df_era5['wind_speed_84m_era5'] * np.sin(wd)

    df_era5['air_density_era5'] = df_era5['pressure_msl_era5'] * 100 / (287.05 * (df_era5['temperature_84m_era5'] + 273.15))
    df_era5['wind_power_density_era5'] = 0.5 * df_era5['air_density_era5'] * df_era5['ws84_cubed_era5']

    ws = df_era5['wind_speed_84m_era5'].values
    theory = np.zeros_like(ws)
    reg = (ws >= 3.0) & (ws < 10.3)
    theory[reg] = 90.09 * ((ws[reg] - 3.0) / (10.3 - 3.0))**3
    rated = (ws >= 10.3) & (ws <= 25.0)
    theory[rated] = 90.09
    df_era5['theory_power_total_era5'] = theory

    for lag in [1, 2, 3]:
        df_era5[f'ws84_lag{lag}_era5'] = df_era5['wind_speed_84m_era5'].shift(lag)
    for lag in [1, 2, 3]:
        wd_lag = df_era5['wind_direction_84m_era5'].shift(lag)
        df_era5[f'wd84_lag{lag}_sin_era5'] = np.sin(2 * np.pi * wd_lag)
        df_era5[f'wd84_lag{lag}_cos_era5'] = np.cos(2 * np.pi * wd_lag)

    df_era5['shear_120_84_era5'] = df_era5['ws100_era5'] - df_era5['wind_speed_84m_era5']
    df_era5['shear_84_10_era5'] = df_era5['wind_speed_84m_era5'] - df_era5['ws10_era5']

    return df_era5

train_era5 = process_era5(train_era5)
valid_era5 = process_era5(valid_era5)

# =============================================================================
# 4. ИНТЕРПОЛЯЦИЯ ИСХОДНЫХ ДАННЫХ НА 84 м + БАЗОВЫЕ ПРИЗНАКИ
# =============================================================================
HUB_H, LOW_H, HIGH_H = 84.0, 80.0, 120.0
for df in [train, valid]:
    ratio = df['wind_speed_120m'] / df['wind_speed_80m'].clip(lower=0.1)
    alpha = np.log(ratio.clip(lower=0.1)) / np.log(HIGH_H / LOW_H)
    df['wind_speed_84m'] = df['wind_speed_80m'] * (HUB_H / LOW_H) ** alpha

    sin_low, cos_low = np.sin(2*np.pi*df['wind_direction_80m']), np.cos(2*np.pi*df['wind_direction_80m'])
    sin_high, cos_high = np.sin(2*np.pi*df['wind_direction_120m']), np.cos(2*np.pi*df['wind_direction_120m'])
    w_low = (HIGH_H - HUB_H) / (HIGH_H - LOW_H)
    sin84 = sin_low * w_low + sin_high * (1 - w_low)
    cos84 = cos_low * w_low + cos_high * (1 - w_low)
    df['wind_direction_84m'] = np.arctan2(sin84, cos84) / (2*np.pi) % 1.0

    if 'temperature_120m' in df.columns:
        df['temperature_84m'] = df['temperature_80m'] * w_low + df['temperature_120m'] * (1 - w_low)
    else:
        df['temperature_84m'] = df['temperature_80m']

def add_base_features(df):
    df = df.copy()
    dt = df[DT_COL]
    df['dayofyear'] = dt.dt.dayofyear
    df['dayofweek'] = dt.dt.dayofweek
    df['week']      = dt.dt.isocalendar().week.astype(int)
    df['hour_sin']  = np.sin(2 * np.pi * df['hour_of_day'] / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour_of_day'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['doy_sin']   = np.sin(2 * np.pi * df['dayofyear'] / 365)
    df['doy_cos']   = np.cos(2 * np.pi * df['dayofyear'] / 365)

    for col in ['wind_direction_10m', 'wind_direction_80m',
                'wind_direction_120m', 'wind_direction_180m']:
        df[col + '_sin'] = np.sin(2 * np.pi * df[col])
        df[col + '_cos'] = np.cos(2 * np.pi * df[col])

    df['wind_speed_80m_sq']    = df['wind_speed_80m'] ** 2
    df['wind_speed_80m_cube']  = df['wind_speed_80m'] ** 3
    df['wind_speed_120m_cube'] = df['wind_speed_120m'] ** 3
    df['wind_shear']           = df['wind_speed_120m'] - df['wind_speed_10m']
    df['gust_ratio']           = df['wind_gusts_10m'] / (df['wind_speed_10m'] + 0.1)
    df['wind_avg']             = df[['wind_speed_80m', 'wind_speed_120m']].mean(axis=1)

    df['ws84_cubed'] = df['wind_speed_84m'] ** 3
    wd = 2 * np.pi * df['wind_direction_84m']
    df['u84'] = df['wind_speed_84m'] * np.cos(wd)
    df['v84'] = df['wind_speed_84m'] * np.sin(wd)

    if 'pressure_msl' in df.columns and 'temperature_84m' in df.columns:
        df['air_density'] = df['pressure_msl'] * 100 / (287.05 * (df['temperature_84m'] + 273.15))
        df['wind_power_density'] = 0.5 * df['air_density'] * df['ws84_cubed']

    if 'wind_speed_120m' in df.columns:
        df['shear_120_84'] = df['wind_speed_120m'] - df['wind_speed_84m']
    if 'wind_speed_10m' in df.columns:
        df['shear_84_10'] = df['wind_speed_84m'] - df['wind_speed_10m']

    ws = df['wind_speed_84m'].values
    theory = np.zeros_like(ws)
    reg = (ws >= 3.0) & (ws < 10.3)
    theory[reg] = 90.09 * ((ws[reg] - 3.0) / (10.3 - 3.0))**3
    rated = (ws >= 10.3) & (ws <= 25.0)
    theory[rated] = 90.09
    df['theory_power_total'] = theory

    for lag in [1, 2, 3]:
        df[f'ws84_lag{lag}'] = df['wind_speed_84m'].shift(lag)
    for lag in [1, 2, 3]:
        wd_lag = df['wind_direction_84m'].shift(lag)
        df[f'wd84_lag{lag}_sin'] = np.sin(2 * np.pi * wd_lag)
        df[f'wd84_lag{lag}_cos'] = np.cos(2 * np.pi * wd_lag)

    df['repair_ma6']  = df['Кол-во_ВЭУ_в_ремонте'].rolling(6, min_periods=1).mean()
    df['has_repair']  = (df['Кол-во_ВЭУ_в_ремонте'] > 0).astype(int)
    df['turbines_available'] = N_TURBINES - df['Кол-во_ВЭУ_в_ремонте']
    df['available_ratio']    = df['turbines_available'] / N_TURBINES
    return df

def fill_missing_180m(df):
    df = df.copy()
    df['wind_speed_180m']         = df['wind_speed_180m'].fillna(df['wind_speed_120m'])
    df['wind_direction_180m']     = df['wind_direction_180m'].fillna(df['wind_direction_120m'])
    df['wind_direction_180m_sin'] = df['wind_direction_180m_sin'].fillna(df['wind_direction_120m_sin'])
    df['wind_direction_180m_cos'] = df['wind_direction_180m_cos'].fillna(df['wind_direction_120m_cos'])
    return df

train_f = fill_missing_180m(add_base_features(train))
valid_f = fill_missing_180m(add_base_features(valid))

# =============================================================================
# 5. ОБЪЕДИНЕНИЕ С ERA5
# =============================================================================
lag_cols = [c for c in train_f.columns if 'lag' in c and not c.endswith('_era5')]
train_f = train_f.dropna(subset=lag_cols).reset_index(drop=True)
valid_f[lag_cols] = valid_f[lag_cols].fillna(train_f[lag_cols].median())

train_era5[DT_COL] = train_era5[DT_COL].astype(train_f[DT_COL].dtype)
valid_era5[DT_COL] = valid_era5[DT_COL].astype(valid_f[DT_COL].dtype)

train_final = train_f.merge(train_era5, on=DT_COL, how='left')
valid_final = valid_f.merge(valid_era5, on=DT_COL, how='left')

era5_cols = [c for c in train_final.columns if c.endswith('_era5')]
train_final[era5_cols] = train_final[era5_cols].fillna(train_final[era5_cols].median())
valid_final[era5_cols] = valid_final[era5_cols].fillna(train_final[era5_cols].median())

# =============================================================================
# 6. ПРИЗНАКИ ДЛЯ ДЕРЕВЬЕВ (ансамбль 8.016%) – используем для стартового заполнения
# =============================================================================
EXCLUDE_FEATURES = [TARGET, DT_COL, 'Кол-во_ВЭУ_в_ремонте']
FEATURES = [c for c in train_final.columns if c not in EXCLUDE_FEATURES]
print(f"Финальное количество признаков: {len(FEATURES)}")

X = train_final[FEATURES].values
y = train_final[TARGET].values
X_valid = valid_final[FEATURES].values

# Обучаем ансамбль 8.016% (чтобы получить стартовые предсказания для нейросети)
lgb_models = []
for seed in SEEDS:
    m = lgb.LGBMRegressor(
        n_estimators=3000, learning_rate=0.025, num_leaves=63,
        min_child_samples=20, subsample=0.85, colsample_bytree=0.85,
        reg_alpha=0.1, reg_lambda=0.1,
        random_state=seed, n_jobs=-1, verbose=-1,
    )
    m.fit(X, y)
    lgb_models.append(m)

rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X, y)

cat = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_seed=42, verbose=False)
cat.fit(X, y, verbose=False)

# Веса из версии 8.016% (можно подставить лучшие, полученные ранее, но для простоты возьмём равные или из предыдущего опыта)
best_weights = np.array([0.95, 0.05, 0.0])  # LGBM, RF, CAT (примерные, т.к. LGBM доминирует)
ensemble_pred = best_weights[0]*np.mean([m.predict(X_valid) for m in lgb_models], axis=0) + \
               best_weights[1]*rf.predict(X_valid) + best_weights[2]*cat.predict(X_valid)
ensemble_pred = np.clip(ensemble_pred, 0, CAPACITY)

# Прогноз ансамбля для train (для заполнения начала окон нейросети)
ensemble_train_pred = best_weights[0]*np.mean([m.predict(X) for m in lgb_models], axis=0) + \
                     best_weights[1]*rf.predict(X) + best_weights[2]*cat.predict(X)
ensemble_train_pred = np.clip(ensemble_train_pred, 0, CAPACITY)

# =============================================================================
# 7. ПОДГОТОВКА ДАННЫХ ДЛЯ НЕЙРОСЕТИ
# =============================================================================
# Масштабируем признаки
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
X_valid_scaled = scaler_X.transform(X_valid)

# Создаём последовательности
def create_sequences(data, target=None, lookback=LOOKBACK):
    Xs = []
    for i in range(lookback, len(data)):
        Xs.append(data[i-lookback:i])
    if target is not None:
        return np.array(Xs), target[lookback:]
    return np.array(Xs)

X_seq, y_seq = create_sequences(X_scaled, y)
X_valid_seq = create_sequences(X_valid_scaled)

# Целевая переменная для нейросети – остаётся как есть (МВт)
# Масштабируем её отдельно
scaler_y = StandardScaler()
y_seq_scaled = scaler_y.fit_transform(y_seq.reshape(-1, 1)).flatten()

# Тензоры
X_train_tensor = torch.tensor(X_seq, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_seq_scaled, dtype=torch.float32).view(-1, 1).to(device)
X_valid_tensor = torch.tensor(X_valid_seq, dtype=torch.float32).to(device)

# =============================================================================
# 8. АРХИТЕКТУРА GRU + Attention
# =============================================================================
class GRUAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers,
                          batch_first=True, bidirectional=True, dropout=dropout)
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, 1),
            nn.Tanh()
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        gru_out, _ = self.gru(x)                 # (batch, seq, hidden*2)
        attn_weights = self.attention(gru_out)   # (batch, seq, 1)
        attn_weights = torch.softmax(attn_weights, dim=1)
        context = torch.sum(gru_out * attn_weights, dim=1)  # (batch, hidden*2)
        context = self.dropout(context)
        out = self.fc(context)
        return out

model = GRUAttention(X_seq.shape[2], hidden_dim=64, num_layers=2, dropout=0.2).to(device)
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# =============================================================================
# 9. ОБУЧЕНИЕ НЕЙРОСЕТИ
# =============================================================================
epochs = 150
batch_size = 64
dataset = TensorDataset(X_train_tensor, y_train_tensor)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print("\nОбучение GRU+Attention...")
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for Xb, yb in loader:
        optimizer.zero_grad()
        preds = model(Xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(Xb)
    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} | loss: {train_loss/len(dataset):.4f}")

# =============================================================================
# 10. ПРОГНОЗ НЕЙРОСЕТИ ДЛЯ VALID (с заполнением начала ансамблем)
# =============================================================================
model.eval()
with torch.no_grad():
    valid_preds_scaled = model(X_valid_tensor).cpu().numpy()
valid_preds_nn = scaler_y.inverse_transform(valid_preds_scaled).flatten()

# Полный массив для валидации
nn_full_preds = np.empty(len(valid_final))
nn_full_preds[:LOOKBACK] = ensemble_pred[:LOOKBACK]  # заполняем ансамблем
nn_full_preds[LOOKBACK:] = valid_preds_nn

# =============================================================================
# 11. БЛЕНДИНГ АНСАМБЛЯ И НЕЙРОСЕТИ НА HOLDOUT
# =============================================================================
# Повторяем out-of-fold для нейросети? Это сложно, поэтому подберём веса на holdout напрямую.
holdout_mask = (train_final[DT_COL] >= '2025-01-01') & (train_final[DT_COL] < '2025-04-01')
# Получим прогноз ансамбля на holdout
ho_ensemble = best_weights[0]*np.mean([m.predict(X[holdout_mask]) for m in lgb_models], axis=0) + \
              best_weights[1]*rf.predict(X[holdout_mask]) + best_weights[2]*cat.predict(X[holdout_mask])
ho_ensemble = np.clip(ho_ensemble, 0, CAPACITY)

# Нейросеть на holdout (обучим быстро на трейне без holdout)
X_ho_nn = X_scaled[holdout_mask]
X_tr_nn = X_scaled[~holdout_mask]
y_tr_nn = y[~holdout_mask]

# Создаём последовательности для обучения нейросети (только на train без holdout)
X_tr_seq, y_tr_seq = create_sequences(X_tr_nn, y_tr_nn)
X_ho_seq = create_sequences(X_ho_nn)

# Масштабируем y
scaler_y_ho = StandardScaler()
y_tr_seq_scaled = scaler_y_ho.fit_transform(y_tr_seq.reshape(-1, 1)).flatten()

# Обучаем маленькую нейросеть (быстро)
model_ho = GRUAttention(X_seq.shape[2], hidden_dim=32, num_layers=1, dropout=0.1).to(device)
optimizer_ho = torch.optim.Adam(model_ho.parameters(), lr=0.001)
criterion_ho = nn.L1Loss()
X_tr_tensor = torch.tensor(X_tr_seq, dtype=torch.float32).to(device)
y_tr_tensor = torch.tensor(y_tr_seq_scaled, dtype=torch.float32).view(-1, 1).to(device)
X_ho_tensor = torch.tensor(X_ho_seq, dtype=torch.float32).to(device)

loader_ho = DataLoader(TensorDataset(X_tr_tensor, y_tr_tensor), batch_size=64, shuffle=True)
for epoch in range(50):
    for Xb, yb in loader_ho:
        optimizer_ho.zero_grad()
        loss = criterion_ho(model_ho(Xb), yb)
        loss.backward()
        optimizer_ho.step()

model_ho.eval()
with torch.no_grad():
    ho_preds_nn_scaled = model_ho(X_ho_tensor).cpu().numpy()
ho_preds_nn = scaler_y_ho.inverse_transform(ho_preds_nn_scaled).flatten()

# Заполняем начало holdout ансамблем
nn_ho_full = np.concatenate([ho_ensemble[:LOOKBACK], ho_preds_nn])

# Подбор весов для блендинга
best_mae = np.inf
best_w_nn = 0.0
for w_nn in np.arange(0, 1.01, 0.05):
    blended = (1 - w_nn) * ho_ensemble + w_nn * nn_ho_full
    mae = mean_absolute_error(y[holdout_mask], blended)
    if mae < best_mae:
        best_mae = mae
        best_w_nn = w_nn
print(f"Оптимальный вес нейросети: {best_w_nn:.2f}, MAE holdout: {best_mae:.3f} МВт ({100*best_mae/CAPACITY:.2f}%)")

# =============================================================================
# 12. ФИНАЛЬНЫЙ ПРОГНОЗ
# =============================================================================
final_pred = (1 - best_w_nn) * ensemble_pred + best_w_nn * nn_full_preds
final_pred = np.clip(final_pred, 0, CAPACITY)

# Защита по доступной мощности
zero_mask = valid_f['turbines_available'].values <= 0
final_pred[zero_mask] = 0.0
final_pred = np.minimum(final_pred, CAPACITY * valid_f['available_ratio'].values + 1e-3)
final_pred = np.clip(final_pred, 0, CAPACITY)

result = (pd.DataFrame({DT_COL: valid_f[DT_COL].values, TARGET: final_pred})
            .set_index(DT_COL).loc[valid_order].reset_index())
result[[TARGET]].to_csv(OUTPUT_VALID, index=False)
print(f"[OK] Сохранён {OUTPUT_VALID} ({len(result)} строк)")

Используемое устройство: cpu
Загрузка ERA5 для train...
Загрузка ERA5 для valid...
Финальное количество признаков: 92


KeyboardInterrupt: 